# HiOCFD 기준 데이터와 Re=1600 소산율 비교

**사용법:** 런타임 → 모두 실행. `results.pkl`이 없으면 업로드 창이 뜹니다(taylor_green_final에서 받은 파일). 기준 파일은 자동으로 내려받습니다(인터넷이 되는 Colab에서만 동작).
끝나면 출력 전체를 복사해서 보내 주세요.

In [ ]:
# ===== 셀 1: results.pkl 올리기 + 기준 파일 받기 =====
import os, pickle, subprocess
import numpy as np
import matplotlib.pyplot as plt

if not os.path.exists("results.pkl"):
    from google.colab import files
    print("results.pkl을 선택해서 올려 주세요 (taylor_green_final에서 받은 파일)")
    files.upload()
results = pickle.load(open("results.pkl", "rb"))
print("불러온 결과:", [(r["N"], int(r["Re"])) for r in results])

REF = "spectral_Re1600_512.gdiag"
URL = "https://cfd.ku.edu/hiocfd/spectral_Re1600_512.gdiag"
if not os.path.exists(REF):
    subprocess.run(["wget", "-q", "-O", REF, URL])
print("기준 파일 크기:", os.path.getsize(REF) if os.path.exists(REF) else "없음", "바이트")
print("--- 파일 앞부분 ---")
print(open(REF, errors="ignore").read()[:700])


In [ ]:
# ===== 셀 2: 열 자동 판별 + 비교 + 그림 + 요약 =====
rows = []
for line in open(REF, errors="ignore"):
    s = line.strip()
    if not s or s.startswith("#"):
        continue
    try:
        vals = [float(x) for x in s.replace(",", " ").split()]
    except ValueError:
        continue
    if len(vals) >= 3:
        rows.append(vals)
w = min(len(r) for r in rows)
ref = np.array([r[:w] for r in rows])
t_ref = ref[:, 0]
print(f"기준 데이터: {ref.shape[0]}행 x {ref.shape[1]}열, t = {t_ref.min():g} ~ {t_ref.max():g}")

LOG = []
def P(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

# 열 판별: E는 t=0에서 0.125(=1/8), 소산율 열은 최댓값이 0.011~0.015
cols = list(range(1, ref.shape[1]))
E_cols = [c for c in cols if abs(ref[0, c] - 0.125) < 0.01]
eps_cols = [c for c in cols if 0.011 < ref[:, c].max() < 0.015]
P("운동에너지로 보이는 열:", E_cols, "| 소산율로 보이는 열:", eps_cols)
if not eps_cols:
    raise SystemExit("소산율 열을 자동으로 못 찾았습니다. 위에 출력된 파일 앞부분을 보내 주세요.")

ours = {r["N"]: r for r in results if abs(r["Re"] - 1600) < 1e-9 and abs(r.get("dtmax", 0.01) - 0.01) < 1e-12}
fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
P("\n" + "=" * 70)
for c in eps_cols:
    e_ref = ref[:, c]; i_ref = int(np.argmax(e_ref))
    P(f"[기준 열 {c}] 피크 ε = {e_ref[i_ref]:.5f} at t = {t_ref[i_ref]:.3f}")
    ax[0].plot(t_ref, e_ref, "k-", lw=2 if c == eps_cols[0] else 1, label=f"reference 512^3 (col {c})")
    for N, r in sorted(ours.items()):
        m = (t_ref >= 0) & (t_ref <= min(10.0, r["t"].max()))
        e_int = np.interp(t_ref[m], r["t"], r["eps"])
        dev = np.abs(e_int - e_ref[m]) / e_ref.max()
        ip = int(np.argmax(r["eps"]))
        P(f"  N={N}: 피크 ε = {r['eps'][ip]:.5f} (기준 대비 {100*(r['eps'][ip]-e_ref[i_ref])/e_ref[i_ref]:+.2f}%), "
          f"피크 시각 {r['t'][ip]:.2f} (기준 {t_ref[i_ref]:.2f}), "
          f"max|Δε|/ε_ref,max = {dev.max():.2e}, rms = {np.sqrt((dev**2).mean()):.2e}")
        if c == eps_cols[0]:
            ax[0].plot(r["t"], r["eps"], "--", label=f"this work N={N}")
            ax[1].plot(t_ref[m], (e_int - e_ref[m]) / e_ref.max(), label=f"N={N}")
ax[0].set_xlabel("t"); ax[0].set_ylabel("dissipation rate"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ax[1].set_xlabel("t"); ax[1].set_ylabel(r"$(\varepsilon-\varepsilon_{ref})/\varepsilon_{ref,max}$"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
ax[1].axhline(0, color="k", lw=.5)
plt.tight_layout(); plt.savefig("fig8_benchmark.png", dpi=150); plt.show()

for c in E_cols[:1]:
    for N, r in sorted(ours.items()):
        m = (t_ref <= min(10.0, r["t"].max()))
        d = np.abs(np.interp(t_ref[m], r["t"], r["E"]) - ref[m, c]).max()
        P(f"  운동에너지 E(t): N={N} max|ΔE| = {d:.2e}")
open("benchmark_summary.txt", "w", encoding="utf-8").write("\n".join(LOG))
try:
    from google.colab import files
    files.download("fig8_benchmark.png"); files.download("benchmark_summary.txt")
except Exception:
    pass
print("\n위 출력(특히 [기준 열 ...] 블록)을 그대로 복사해서 보내 주세요.")
